In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
user_credential = user_secrets.get_gcloud_credential()
user_secrets.set_tensorflow_credential(user_credential)
user_secrets.set_gcloud_credentials("mobilewaft" ,user_credential)


In [ ]:
from __future__ import annotations

import gzip
import io
import json
import random
import concurrent.futures
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional
import os
import shutil
import time
import zipfile
import threading

import numpy as np
import boto3
from botocore.config import Config
from google.cloud import storage
import dotenv
dotenv.load_dotenv()


# ─────────────────────────────────────────────
# Config
# ─────────────────────────────────────────────

@dataclass
class SamplerConfig:
    bucket_name: str = "gresearch"
    prefix: str = "sanpo_dataset/v0/sanpo-real/"
    max_sessions: int = 500
    batch_size: int = 50
    max_sample: int = 100        # frame triples per session per camera
    stride: int = 15             # lấy 1 frame mỗi N frames để tránh near-duplicate
    seed: int = 42
    cameras: List[str] = field(default_factory=lambda: ["head", "chest"])
    require_all_modalities: bool = True
    size_limit_gb: float = 100.0


# ─────────────────────────────────────────────
# Sampler
# ─────────────────────────────────────────────

class SanpoDepthPairSampler:
    def __init__(self, config: SamplerConfig):
        self.cfg = config
        self.client = storage.Client.create_anonymous_client()
        self.rng = random.Random(config.seed)

    def _gs_uri(self, object_name: str) -> str:
        return f"gs://{self.cfg.bucket_name}/{object_name}"

    def _session_prefix(self, session_id: str) -> str:
        return f"{self.cfg.prefix.rstrip('/')}/{session_id}/"

    def _calib_url(self, session_id: str) -> str:
        return self._gs_uri(f"{self._session_prefix(session_id)}description.json")

    def _list_prefixes(self, prefix: str) -> List[str]:
        it = self.client.list_blobs(
            self.cfg.bucket_name, prefix=prefix, delimiter="/"
        )
        prefixes = set()
        for page in it.pages:
            prefixes.update(page.prefixes)
        return sorted(prefixes)

    def _list_files_flat(self, prefix: str) -> List[str]:
        it = self.client.list_blobs(
            self.cfg.bucket_name, prefix=prefix, delimiter="/"
        )
        out = []
        for page in it.pages:
            for blob in page:
                base = blob.name.rsplit("/", 1)[-1]
                if base and not base.endswith("$folder$"):
                    out.append(blob.name)
        return sorted(out)

    def _normalize_id(self, path: str) -> str:
        name = path.rsplit("/", 1)[-1]
        if name.endswith(".float16.gz"): return name[:-11]
        return name.rsplit(".", 1)[0] if "." in name else name

    def _frame_map(self, session_id: str, rel_prefix: str) -> Dict[str, str]:
        prefix = f"{self._session_prefix(session_id)}{rel_prefix.strip('/')}/"
        files = self._list_files_flat(prefix)
        return {self._normalize_id(f): f for f in files}

    def _sample_camera(self, left_map, right_map, depth_map) -> List[dict]:
        sets = [set(m.keys()) for m in (left_map, right_map, depth_map) if m]
        if not sets: return []
        valid_ids = sorted(
            set.intersection(*sets) if self.cfg.require_all_modalities
            else set.union(*sets)
        )
        if not valid_ids: return []
        strided = valid_ids[:: max(1, self.cfg.stride)]
        chosen = self.rng.sample(strided, k=min(self.cfg.max_sample, len(strided)))
        return [
            {
                "id": fid,
                "left":  self._gs_uri(left_map[fid])  if fid in left_map  else None,
                "right": self._gs_uri(right_map[fid]) if fid in right_map else None,
                "depth": self._gs_uri(depth_map[fid]) if fid in depth_map else None,
            }
            for fid in chosen
        ]

    def dry_run(self, ignored_sessions: Optional[List[str]] = None, num_sessions_to_select: Optional[int] = None) -> List[dict]:
        """
        Args:
            ignored_sessions: List session_id gốc (hash) đã processed.
            num_sessions_to_select: Số lượng session cần select cho batch này.
        """
        all_prefixes = self._list_prefixes(self.cfg.prefix.rstrip("/") + "/")
        session_ids = [p.rstrip("/").split("/")[-1] for p in all_prefixes]
        if not session_ids: return []

        ignored_set = set(ignored_sessions or [])
        available = [s for s in session_ids if s not in ignored_set]
        if not available:
            print("[sampler] Tất cả session đã được download.")
            return []

        k = num_sessions_to_select or self.cfg.batch_size
        chosen = self.rng.sample(available, k=min(k, len(available)))
        CAM_DIRS = {"head": "camera_head", "chest": "camera_chest"}
        results = []

        for sid in chosen:
            entry = {
                "session_name": sid,
                "calib_url": self._calib_url(sid),
            }
            for cam in self.cfg.cameras:
                cdir = CAM_DIRS[cam]
                entry[cam] = self._sample_camera(
                    self._frame_map(sid, f"{cdir}/left/video_frames"),
                    self._frame_map(sid, f"{cdir}/right/video_frames"),
                    self._frame_map(sid, f"{cdir}/left/depth_maps"),
                )
            results.append(entry)

        total = sum(len(r.get(c, [])) for r in results for c in self.cfg.cameras)
        print(f"[sampler] Chosen {len(results)} sessions | {total} frame triples")
        return results


# ─────────────────────────────────────────────
# Helpers: Calib + Depth
# ─────────────────────────────────────────────

def _extract_calib(desc: dict, camera: str) -> dict:
    cam_key = "camera_head" if camera == "head" else "camera_chest"
    locations = desc.get("session_camera_location", [])
    details   = desc.get("session_camera_details", [])
    cam_detail = next(
        (details[i] for i, loc in enumerate(locations) if loc == cam_key and i < len(details)),
        details[1 if camera == "head" else 0] if details else None,
    )
    if not cam_detail: return {}
    lp = cam_detail["left_camera_params"]
    return {
        "camera":           cam_key,
        "focal_length_px":  lp["fx"],
        "baseline_m":       round(abs(cam_detail["stereo_transform"]["coeff"][3]) / 1000.0, 8),
        "cx":               lp["cx"],
        "cy":               lp["cy"],
        "image_width":      lp["image_width"],
        "image_height":     lp["image_height"],
        "fps":              cam_detail.get("fps"),
        "model":            cam_detail.get("model"),
    }


def _decode_float16_gz(raw: bytes) -> np.ndarray:
    with gzip.open(io.BytesIO(raw), "rb") as f:
        data = np.frombuffer(f.read(), dtype=np.float16)
    h = int(data[0])
    w = int(data[1])
    return data[2:].reshape(h, w).astype(np.float32)


# ─────────────────────────────────────────────
# Session Index Helpers
# ─────────────────────────────────────────────

def _load_session_index(output_dir: str) -> Dict[str, str]:
    index_file = Path(output_dir) / "session_index.json"
    if index_file.exists():
        with open(index_file, "r") as f:
            return json.load(f)
    return {}


def _save_session_index(output_dir: str, index: Dict[str, str]) -> None:
    index_file = Path(output_dir) / "session_index.json"
    index_file.parent.mkdir(parents=True, exist_ok=True)
    with open(index_file, "w") as f:
        json.dump(index, f, indent=2)


def _next_session_number(index: Dict[str, str]) -> int:
    if not index:
        return 1
    existing = [int(v.split("_")[1]) for v in index.values() if "_" in v]
    return max(existing) + 1 if existing else 1


def _load_processed_sessions(output_dir: str) -> List[str]:
    p_file = Path(output_dir) / "processed_sessions.json"
    if p_file.exists():
        with open(p_file, "r") as f:
            return json.load(f)
    return []


def _save_processed_sessions(output_dir: str, processed: List[str]) -> None:
    p_file = Path(output_dir) / "processed_sessions.json"
    p_file.parent.mkdir(parents=True, exist_ok=True)
    with open(p_file, "w") as f:
        json.dump(processed, f, indent=2)


# ─────────────────────────────────────────────
# Safe Counter for Progress Tracking
# ─────────────────────────────────────────────

class SafeCounter:
    def __init__(self):
        self.value = 0
        self.lock = threading.Lock()
    def add(self, n):
        with self.lock:
            self.value += n
    def get(self):
        with self.lock:
            return self.value


# ─────────────────────────────────────────────
# Downloader with expected folder structure
# ─────────────────────────────────────────────

def download_sanpo_dataset(
    results: List[dict],
    bucket_name: str = "gresearch",
    output_dir: str = "/kaggle/working/sanpo_real",
    max_workers: int = 8,
    cameras: Optional[List[str]] = None,
    counter: Optional[SafeCounter] = None,
) -> None:
    """
    Folder structure expected to train the model:
        {output_dir}/
        ├── session_index.json
        ├── session_0001/
        │   ├── left/            RGB PNGs
        │   ├── right/           RGB PNGs
        │   ├── depth_ml/        float32 .npy (meters)
        │   └── calib.json
        ├── session_0002/
        └── ...
    """
    if cameras is None:
        cameras = [c for c in ["head", "chest"] if any(c in r for r in results)]

    client = storage.Client.create_anonymous_client()
    root = Path(output_dir)
    root.mkdir(parents=True, exist_ok=True)

    def fetch(uri: str) -> bytes:
        obj = uri[len(f"gs://{bucket_name}/"):]
        return client.bucket(bucket_name).blob(obj).download_as_bytes()

    # ── Assign session_XXX numbers per (session, camera) ──────────────────────────────────
    index = _load_session_index(output_dir)
    next_num = _next_session_number(index)

    for s in results:
        sid = s["session_name"]
        for cam in cameras:
            if not s.get(cam):
                continue
            key = f"{sid}_{cam}"
            if key not in index:
                index[key] = f"session_{next_num:04d}"
                next_num += 1

    _save_session_index(output_dir, index)

    # ── Write calib.json per session_XXX folder ───────────────────────────────────
    for s in results:
        sid = s["session_name"]
        desc = None
        if s.get("calib_url"):
            try:
                desc = json.loads(fetch(s["calib_url"]))
            except Exception as e:
                print(f"[WARN] calib fetch failed {sid}: {e}")

        for cam in cameras:
            if not s.get(cam):
                continue
            key = f"{sid}_{cam}"
            folder = index[key]
            session_dir = root / folder
            for d in ("left", "right", "depth_ml"):
                (session_dir / d).mkdir(parents=True, exist_ok=True)
            if desc:
                calib = _extract_calib(desc, cam)
                (session_dir / "calib.json").write_text(json.dumps(calib, indent=2))

    # ── Build task list ────────────────────────────────────────────────────────
    tasks = []
    for s in results:
        sid = s["session_name"]
        for cam in cameras:
            if not s.get(cam):
                continue
            key = f"{sid}_{cam}"
            folder = index[key]
            session_dir = root / folder
            for frame in s.get(cam, []):
                tasks.append((frame, session_dir))

    total = len(tasks)
    print(f"[download] {total} frames | {max_workers} workers")

    # ── Parallel download ──────────────────────────────────────────────────────
    done = errors = 0

    def process(task):
        nonlocal done, errors
        frame, base = task
        fid = frame["id"]
        downloaded_bytes = 0
        try:
            if frame.get("left"):
                left_data = fetch(frame["left"])
                (base / "left"  / f"{fid}.png").write_bytes(left_data)
                downloaded_bytes += len(left_data)
            if frame.get("right"):
                right_data = fetch(frame["right"])
                (base / "right" / f"{fid}.png").write_bytes(right_data)
                downloaded_bytes += len(right_data)
            if frame.get("depth"):
                depth_data = fetch(frame["depth"])
                arr = _decode_float16_gz(depth_data)
                np.save(str(base / "depth_ml" / f"{fid}.npy"), arr)
                downloaded_bytes += len(depth_data)
            
            if counter:
                counter.add(downloaded_bytes)
        except Exception as e:
            print(f"[ERROR] failed to download/process frame {fid}: {e}")
            raise e

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = {pool.submit(process, t): t for t in tasks}
        for fut in concurrent.futures.as_completed(futures):
            done += 1
            if fut.exception():
                errors += 1
            if done % 100 == 0 or done == total:
                print(f"[progress] {done}/{total}  errors={errors}")

    print(f"[done] {done - errors}/{total} frames downloaded.")


# ─────────────────────────────────────────────
# S3-compatible R2 Client
# ─────────────────────────────────────────────

def get_r2_client():
    r2_access_key = os.environ.get('CF_R2_ACCESS_KEY_ID')
    r2_secret_key = os.environ.get('CF_R2_SECRET_ACCESS_KEY')
    r2_endpoint = os.environ.get('CF_R2_ENDPOINT_URL')
    
    if not (r2_access_key and r2_secret_key and r2_endpoint):
        try:
            from kaggle_secrets import UserSecretsClient
            user_secrets = UserSecretsClient()
            r2_access_key = user_secrets.get_secret('CF_R2_ACCESS_KEY_ID')
            r2_secret_key = user_secrets.get_secret('CF_R2_SECRET_ACCESS_KEY')
            r2_endpoint = user_secrets.get_secret('CF_R2_ENDPOINT_URL')
        except Exception as e:
            raise ValueError(f"Credentials not found: {e}")

    s3_client = boto3.client(
        service_name='s3',
        endpoint_url=r2_endpoint,
        aws_access_key_id=r2_access_key,
        aws_secret_access_key=r2_secret_key,
        config=Config(signature_version='s3v4')
    )
    return s3_client


def upload_file_to_r2(local_path: str, bucket_name: str, object_name: str):
    s3 = get_r2_client()
    print(f"Uploading {local_path} to R2 bucket {bucket_name} as {object_name}...")
    file_size = os.path.getsize(local_path)
    uploaded = 0
    
    def progress_callback(bytes_amount):
        nonlocal uploaded
        uploaded += bytes_amount
        percentage = (uploaded / file_size) * 100
        print(f"\rUpload progress: {percentage:.2f}% ({uploaded / 1024**2:.2f} MB / {file_size / 1024**2:.2f} MB)", end="", flush=True)

    s3.upload_file(
        Filename=local_path,
        Bucket=bucket_name,
        Key=object_name,
        Callback=progress_callback
    )
    print("\nUpload completed successfully!")


def clean_batch_folders(output_dir: str):
    root = Path(output_dir)
    print(f"Cleaning up session folders in {output_dir}...")
    for item in root.iterdir():
        if item.is_dir() and item.name.startswith("session_"):
            shutil.rmtree(item)


In [ ]:
import math
from IPython.display import clear_output

# ── Configuration ─────────────────────────────────────────────────────────
MAX_SESSIONS = 500       # Total sessions to sample
BATCH_SIZE = 50          # Batch size to download, zip, and upload
SIZE_LIMIT_GB = 100.0    # safety size limit in GB
OUTPUT_DIR = "/kaggle/working/sanpo_real"
ZIP_OUTPUT_DIR = "/kaggle/working/sanpo_real_zips"

cfg = SamplerConfig(
    max_sessions=MAX_SESSIONS,
    batch_size=BATCH_SIZE,
    max_sample=100,
    stride=15,
    cameras=["head", "chest"],
    size_limit_gb=SIZE_LIMIT_GB
)

sampler = SanpoDepthPairSampler(cfg)
root_path = Path(OUTPUT_DIR)
zip_root = Path(ZIP_OUTPUT_DIR)
zip_root.mkdir(parents=True, exist_ok=True)

# Load ignored/processed session list
processed_sessions = _load_processed_sessions(OUTPUT_DIR)
print(f"Already processed: {len(processed_sessions)} sessions.")

# Resolve bucket name
bucket_name = os.environ.get('CF_R2_BUCKET_NAME')
if not bucket_name:
    try:
        user_secrets = UserSecretsClient()
        bucket_name = user_secrets.get_secret('CF_R2_BUCKET_NAME')
    except:
        bucket_name = "waft-stereo"

total_processed_bytes = 0
batch_idx = len(processed_sessions) // BATCH_SIZE + 1
batch_statuses = [f"Batch {i+1:02d}: COMPLETED (Restored from index)" for i in range(len(processed_sessions) // BATCH_SIZE)]

while len(processed_sessions) < MAX_SESSIONS:
    total_processed_gb = total_processed_bytes / (1024**3)
    if total_processed_gb >= SIZE_LIMIT_GB:
        print(f"Safety limit reached: {total_processed_gb:.2f} GB / {SIZE_LIMIT_GB:.2f} GB. Stopping.")
        break
        
    sessions_remaining = MAX_SESSIONS - len(processed_sessions)
    if sessions_remaining <= 0:
        break
        
    num_sessions_to_select = min(BATCH_SIZE, sessions_remaining)
    
    # Render Dashboard
    clear_output(wait=True)
    print("============================================================")
    print("           SANPO REAL BATCH PROCESSOR STATUS")
    print("============================================================")
    print(f"Configured Max Sessions : {MAX_SESSIONS} (Batch Size: {BATCH_SIZE})")
    print(f"Size Limit              : {SIZE_LIMIT_GB:.2f} GB")
    print(f"Processed Sessions      : {len(processed_sessions)} / {MAX_SESSIONS}")
    print(f"Total Uploaded Size     : {total_processed_gb:.4f} GB")
    print("------------------------------------------------------------")
    print("Batch History:")
    for status in batch_statuses:
        print(f"  - {status}")
    print("------------------------------------------------------------")
    print(f"Current Batch {batch_idx:02d} (Selecting {num_sessions_to_select} sessions)...")
    print("============================================================\n")
    
    # 1. Sample sessions for this batch
    results = sampler.dry_run(ignored_sessions=processed_sessions, num_sessions_to_select=num_sessions_to_select)
    if not results:
        print("No more sessions available to download.")
        break
        
    # 2. Download the batch
    print(f"\n[Batch {batch_idx:02d}] Downloading...")
    counter = SafeCounter()
    download_sanpo_dataset(
        results=results,
        bucket_name=cfg.bucket_name,
        output_dir=OUTPUT_DIR,
        max_workers=8,
        counter=counter
    )
    
    # 3. Zip the downloaded batch session folders
    timestamp = int(time.time())
    zip_filename = f"sanpo_real_batch_{batch_idx:02d}_{timestamp}.zip"
    zip_path = zip_root / zip_filename
    
    print(f"\n[Batch {batch_idx:02d}] Compressing batch...")
    session_dirs = [d for d in root_path.iterdir() if d.is_dir() and d.name.startswith("session_")]
    
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for sdir in session_dirs:
            for folderName, subfolders, filenames in os.walk(sdir):
                for filename in filenames:
                    filePath = os.path.join(folderName, filename)
                    relPath = os.path.relpath(filePath, root_path)
                    zipf.write(filePath, relPath)
                    
    zip_size_bytes = os.path.getsize(zip_path)
    zip_size_gb = zip_size_bytes / (1024**3)
    total_processed_bytes += zip_size_bytes
    
    # 4. Upload zip to R2
    print(f"\n[Batch {batch_idx:02d}] Uploading {zip_filename} to R2...")
    try:
        upload_file_to_r2(str(zip_path), bucket_name, f"sanpo_real/{zip_filename}")
        upload_success = True
    except Exception as e:
        print(f"[ERROR] Upload failed for batch {batch_idx:02d}: {e}")
        upload_success = False
        
    # 5. Clean disk space
    print(f"\n[Batch {batch_idx:02d}] Cleaning disk space...")
    clean_batch_folders(OUTPUT_DIR)
    if zip_path.exists():
        zip_path.unlink()
        
    # 6. Record progress
    if upload_success:
        for s in results:
            processed_sessions.append(s["session_name"])
        _save_processed_sessions(OUTPUT_DIR, processed_sessions)
        status_msg = f"Batch {batch_idx:02d}: COMPLETED (Uploaded {zip_filename}, {zip_size_gb:.4f} GB)"
    else:
        status_msg = f"Batch {batch_idx:02d}: FAILED (Upload error)"
        
    batch_statuses.append(status_msg)
    batch_idx += 1

# Final status print
clear_output(wait=True)
print("============================================================")
print("           SANPO REAL BATCH PROCESSOR COMPLETED")
print("============================================================")
print(f"Processed Sessions      : {len(processed_sessions)} / {MAX_SESSIONS}")
print(f"Total Uploaded Size     : {total_processed_bytes / (1024**3):.4f} GB")
print("------------------------------------------------------------")
print("Final Batch Statuses:")
for status in batch_statuses:
    print(f"  - {status}")
print("============================================================")


In [ ]:
def visualize_session(
    output_dir: str,
    session_folder: Optional[str] = None,
    n_frames: int = 3,
) -> None:
    """
    Hiển thị n_frames đầu tiên của session: left | right | depth side-by-side.
    """
    try:
        import matplotlib.pyplot as plt
        import matplotlib.image as mpimg
    except ImportError:
        print("[visualize] Cần cài matplotlib: pip install matplotlib")
        return

    root = Path(output_dir)

    if session_folder is None:
        index = _load_session_index(output_dir)
        if not index:
            print("[visualize] Chưa có session nào trong index.")
            return
        session_folder = sorted(index.values())[0]

    sess_dir = root / session_folder
    if not sess_dir.exists():
        print(f"[visualize] Không tìm thấy session folder: {sess_dir}")
        return

    left_files  = sorted((sess_dir / "left").glob("*.png"))[:n_frames]
    right_files = sorted((sess_dir / "right").glob("*.png"))[:n_frames]
    depth_files = sorted((sess_dir / "depth_ml").glob("*.npy"))[:n_frames]

    n = min(len(left_files), len(right_files), len(depth_files))
    if n == 0:
        print("[visualize] Không tìm thấy file ảnh/depth trong session.")
        return

    fig, axes = plt.subplots(n, 3, figsize=(15, 5 * n))
    if n == 1:
        axes = [axes]

    fig.suptitle(f"{session_folder}", fontsize=14, fontweight="bold")

    for i in range(n):
        left_img  = mpimg.imread(str(left_files[i]))
        right_img = mpimg.imread(str(right_files[i]))
        depth_arr = np.load(str(depth_files[i]))

        # Left RGB
        axes[i][0].imshow(left_img)
        axes[i][0].set_title(f"Left  [{left_files[i].name}]", fontsize=9)
        axes[i][0].axis("off")

        # Right RGB
        axes[i][1].imshow(right_img)
        axes[i][1].set_title(f"Right [{right_files[i].name}]", fontsize=9)
        axes[i][1].axis("off")

        # Depth
        vmin = np.percentile(depth_arr[depth_arr > 0], 2)  if np.any(depth_arr > 0) else 0
        vmax = np.percentile(depth_arr[depth_arr > 0], 98) if np.any(depth_arr > 0) else 1
        im = axes[i][2].imshow(depth_arr, cmap="plasma", vmin=vmax, vmax=vmin)
        axes[i][2].set_title(f"Depth (m) [{depth_files[i].name}]", fontsize=9)
        axes[i][2].axis("off")
        plt.colorbar(im, ax=axes[i][2], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()

print("Visualizer ready. If you download a batch without deleting it, call: visualize_session(OUTPUT_DIR, session_folder='session_0001')")
